# nanovitok 03 — retrofit adaptation (GPU T4 ×2)

**Settings:** Accelerator = GPU T4 ×2, Internet = On. Attach `vitok-code` (built by `vitok.kaggle_session.build_bundle`:
the repository at one commit, plus `retrofit/` with the retrofit tokenizers, their reports, the held-out evaluation files
the extension's check texts and `val_docs.jsonl`, the merge-learning text) and `vitok-data` (training shards,
`test.jsonl`).

Run with **Save Version → Save & Run All**. Each GPU runs its queue of conditions (`vitok.pipeline`): embedding
extension → scoring before training → adaptation training → scoring of snapshots and final weights. Outputs land in
`/kaggle/working/runs/`; every step resumes from its own checkpoint or results file.

| Session (PLAN.md) | `STAGE` | Queues |
|---|---|---|
| smoke run = new-token probe (Stage 6) | `"probe"` | GPU 0 `multisyllable-1000`, GPU 1 `multisyllable` |
| full runs (Stage 7) | `"full"` | from the throughput measured by the probe |

Every code cell calls functions tested on CPU (`vitok.kaggle_session`, `vitok.pipeline`); tests/test_notebook03.py runs
these cells end to end with a tiny model before any push.

In [ ]:
import os
from pathlib import Path
STAGE = "probe"
QUEUES = {0: ["multisyllable-1000"], 1: ["multisyllable"]}   # GPU -> conditions, run one after another
# every step checkpoints and stops at this budget; the probe is capped at 2 h, so it costs at most 2 quota-hours
SESSION_MINUTES = 120 if STAGE == "probe" else 11 * 60 + 30
STEPS = {"probe": 50, "full": None}[STAGE]   # full: set from the throughput the probe measured
SNAPSHOT_STEPS = []              # full: the recovery-curve points
DEV_SHARD, DEV_DOCS = "shard_00019.parquet", 500   # probe dev documents, held out from every run (PLAN.md, Protocol)
SPEED_PROMPTS = 0 if STAGE == "probe" else 20
MODEL, REVISION = "Qwen/Qwen3-0.6B", "c1899de289a04d12100db370d81485cdf75e47ca"
DTYPE = "fp16"                   # the T4 has no bf16
BATCH, MICRO_BATCH, SEQ_LEN = 32, 4, 512   # micro-batch: the probe measures T4 memory; the laptop needed 6 GiB at 1
MINIMUM = {"torch": "2.4", "transformers": "4.56", "tokenizers": "0.21"}
INPUT = Path(os.environ.get("VITOK_KAGGLE_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("VITOK_KAGGLE_WORK", "/kaggle/working/runs"))

In [ ]:
import importlib.util, json, shutil, subprocess, sys, time
START = time.monotonic()
bundle = next(p.parent for p in INPUT.rglob("pyproject.toml") if (p.parent / "src" / "vitok").is_dir())
if importlib.util.find_spec("vitok") is None:   # on Kaggle: install the bundled code
    CODE = Path("/tmp/vitok")
    shutil.copytree(bundle, CODE, dirs_exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(CODE)], check=True)
    sys.path.insert(0, str(CODE / "src"))
from vitok import kaggle_session as ks
inputs = ks.find_inputs(INPUT)
R, D = inputs["code"] / "retrofit", inputs["data"]
print(inputs, json.loads((R / "bundle_manifest.json").read_text())["commit"])

In [ ]:
# Preflight: library versions, idle GPUs, the tokenizers reproduce their measured compression (manipulation check),
# then the pinned base model, after which the queues run offline
from importlib.metadata import version
ks.check_versions({p: version(p) for p in MINIMUM}, MINIMUM)
print({p: version(p) for p in MINIMUM})
print(ks.preflight(need_gpus=len([q for q in QUEUES.values() if q])))
test_docs = [json.loads(line)["text"] for line in (D / "test.jsonl").read_text(encoding="utf-8").splitlines()]
names = ["base", "syllable", "multisyllable", "syllable-1000", "multisyllable-1000"]
print(ks.check_tokenizers({n: R / n / "tokenizer.json" for n in names}, test_docs, ks.expected_compression(R)))
print(ks.prefetch(MODEL, REVISION))

In [ ]:
# The run's configuration: identical training and scoring settings for every condition (PLAN.md, Protocol)
assert STEPS, "set STEPS for a full run (from the probe's measured throughput)"
WORK.mkdir(parents=True, exist_ok=True)
dev = WORK / "dev_docs.jsonl"
dev.write_text("".join(json.dumps({"text": t}, ensure_ascii=False) + "\n"
                       for t in ks.dev_docs(D / "shards" / DEV_SHARD, DEV_DOCS)), encoding="utf-8")
cfg = {"model": MODEL, "revision": REVISION, "base_tokenizer": str(R / "base" / "tokenizer.json"),
       "check_texts": str(R / "check_texts.jsonl"),
       "conditions": {n: {"tokenizer": str(R / n / "tokenizer.json")} for n in names},
       "train": {"steps": STEPS, "batch": BATCH, "micro_batch": MICRO_BATCH, "seq_len": SEQ_LEN, "lr": 1e-4,
                 "warmup": max(1, STEPS // 20), "layers": "0,1,-2,-1", "dtype": DTYPE, "grad_checkpointing": True,
                 "save_every": 25, "log_every": 5, "seed": 0, "snapshot_steps": SNAPSHOT_STEPS},
       "score": {"vi_docs": str(dev if STAGE == "probe" else D / "test.jsonl"),
                 **({} if STAGE == "probe" else {"en_docs": str(R / "en_docs.jsonl"),
                                                  "belebele": str(R / "belebele_vie_Latn.jsonl")}),
                 "batch_size": 8, "dtype": DTYPE, "before_training": STAGE != "probe", "snapshots": True,
                 "speed_prompts": SPEED_PROMPTS},
       "shards": [str(p) for p in sorted((D / "shards").glob("shard_0*.parquet")) if p.name != "shard_99999.parquet"],
       "held_out": [str(D / "test.jsonl"), str(R / "val_docs.jsonl"), str(dev)]}   # val_docs: the merge-learning text
(WORK / f"config-{STAGE}.json").write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg["train"]), len(cfg["shards"]), "shards")

In [ ]:
left = SESSION_MINUTES - (time.monotonic() - START) / 60
codes = ks.launch(QUEUES, WORK / f"config-{STAGE}.json", WORK, deadline_minutes=left)
print("exit codes per GPU:", codes, "(0 done, 3 stopped at the deadline with checkpoints, 1 a condition failed)")

In [ ]:
# Report: throughput, memory, loss and scores, from the files the steps wrote; then every queue log in full
print(json.dumps(ks.report(WORK, [c for q in QUEUES.values() for c in q]), indent=2))
for log in sorted(WORK.glob("queue-*")):
    print("=" * 20, log.name)
    print(log.read_text())
probe_runs = ("multisyllable-1000", "multisyllable")
if STAGE == "probe" and all((WORK / c / "score-final" / "summary.json").exists() for c in probe_runs):
    from vitok import compare
    cmp = compare.compare(WORK / "multisyllable-1000" / "score-final", WORK / "multisyllable" / "score-final")
    (WORK / "probe-comparison.json").write_text(json.dumps(cmp, indent=2))
    print("probe:", json.dumps(cmp["vi"]), "-> choice:", ks.probe_choice(cmp))